# Problems & Goal

- Bài toán cốt lõi: Phân loại đơn nhãn (single-label) 6 lớp theo đúng benchmark. Mặc dù thực tế các lớp có sự chồng chéo ngữ nghĩa (ví dụ: ảnh Tết thường bao gồm cảnh tụ họp), ta sẽ xử lý sự mơ hồ này trực tiếp ở khâu huấn luyện và phân tích lỗi thay vì thay đổi định nghĩa bài toán.
- Vai trò của lớp "other": Hoạt động như một phễu lọc (catch-all failure mode) để hứng các mẫu ngoại lai hoặc không rõ ràng, chứ không mang một đặc trưng ngữ nghĩa độc lập.
- Thách thức từ dữ liệu: Kích thước tập mẫu cực nhỏ, mất cân bằng nghiêm trọng, nhãn nhiễu và ranh giới phân loại mờ nhạt (ví dụ: ảnh công viên dễ nhầm thành thiên nhiên).
- Tiêu chí tối ưu: Ưu tiên tính mạnh mẽ (robustness) và độ tin cậy khi đưa vào thực tế. Không chạy đua tối ưu độ chính xác điểm (point accuracy) trên tập dữ liệu nhỏ vốn rất dễ bị overfit.
- Chiến lược thực thi: Tiếp cận theo hướng tối ưu dữ liệu (data-centric) thay vì dùng mạng end-to-end phức tạp. Giải pháp là sử dụng Frozen Embeddings (SigLIP2 đa ngôn ngữ) kết hợp Classification Head có trọng số, đi kèm kỹ thuật hiệu chuẩn (calibration) và lọc nhiễu offline (CLIPCleaner).

# EDA

## Manifest

In [ ]:
from pathlib import Path

from z_photos.conf import Config

cfg = Config(
    data_root=Path("../data"),
    seed=42,
)

[No output generated]

## Bronze: FiftyOne Dataset

TODO(Explain why FiftyOne)

In [ ]:
from z_photos.datasets import build_bronze_dataset

bronze_dataset = build_bronze_dataset("z_photos-bronze", bronze_dir=cfg.bronze_dir)

[No output generated]

## Inventory & Sanity

1. Class distribution: mất cân bằng giữa train/test?
1. Metadata sanity: kích thước ảnh, file corrupt?
1. Exact duplicates: file trùng lặp chính xác (hash).

In [ ]:
from collections import Counter

import numpy as np
import pandas as pd

# Class distribution per split
train_view = bronze_dataset.match_tags("train")
test_view = bronze_dataset.match_tags("test")

train_counts = Counter(train_view.values("ground_truth.label"))
test_counts = Counter(test_view.values("ground_truth.label"))

all_classes = sorted(set(train_counts) | set(test_counts))
df_dist = pd.DataFrame(
    {
        "class": all_classes,
        "train": [train_counts.get(c, 0) for c in all_classes],
        "test": [test_counts.get(c, 0) for c in all_classes],
    }
).set_index("class")
df_dist["total"] = df_dist["train"] + df_dist["test"]
df_dist["train_pct"] = (df_dist["train"] / df_dist["train"].sum() * 100).round(1)
df_dist["test_pct"] = (df_dist["test"] / df_dist["test"].sum() * 100).round(1)

print("= Class Distribution")
print(f"Train total: {df_dist['train'].sum()}")
print(f"Test total: {df_dist['test'].sum()}")
df_dist.index.name = None
df_dist.style.format({"train_pct": "{:.1f}%", "test_pct": "{:.1f}%"})

= Class Distribution
Train total: 261
Test total: 60


In [ ]:
widths = bronze_dataset.values("metadata.width")
heights = bronze_dataset.values("metadata.height")
sizes = bronze_dataset.values("metadata.size_bytes")

missing_meta = sum(1 for w in widths if w is None)
print(f"Samples với metadata thiếu (có thể corrupt): {missing_meta}")

w_arr = np.array([w for w in widths if w], dtype=float)
h_arr = np.array([h for h in heights if h], dtype=float)
s_arr = np.array([s for s in sizes if s], dtype=float)
ar_arr = w_arr / h_arr

df_meta = pd.DataFrame(
    {
        "metric": ["Width (px)", "Height (px)", "Aspect ratio", "File size (KB)"],
        "min": [w_arr.min(), h_arr.min(), ar_arr.min(), s_arr.min() / 1024],
        "max": [w_arr.max(), h_arr.max(), ar_arr.max(), s_arr.max() / 1024],
        "mean": [w_arr.mean(), h_arr.mean(), ar_arr.mean(), s_arr.mean() / 1024],
        "std": [w_arr.std(), h_arr.std(), ar_arr.std(), s_arr.std() / 1024],
    }
).set_index("metric")
df_meta.index.name = None
df_meta.style.format("{:.1f}")

Samples với metadata thiếu (có thể corrupt): 0


,min,max,mean,std
Width (px),194.0,3464.0,989.5,483.3
Height (px),168.0,3072.0,694.9,379.0
Aspect ratio,0.7,3.2,1.5,0.3
File size (KB),9.6,4212.4,314.6,490.3


## Exact Duplicates

`fob.compute_exact_duplicates`: so sánh hash file MD5/SHA, phát hiện file trùng chính xác. Không cần model.

In [ ]:
import fiftyone.brain as fob
import pandas as pd

exact_dups = fob.compute_exact_duplicates(bronze_dataset, progress=False)

rows = []
for rep_id, dup_ids in list(exact_dups.items())[:5]:
    rep = bronze_dataset[rep_id]
    for did in dup_ids:
        s = bronze_dataset[did]
        rows.append(
            {
                "Rep Label": rep.ground_truth.label,
                "Rep File": rep.filepath.split("/")[-1],
                "Dup Label": s.ground_truth.label,
                "Dup File": s.filepath.split("/")[-1],
            }
        )

pd.DataFrame(rows)

Computing filehashes...


        Rep Label               Rep File Dup Label                   Dup File
0    baby_playing  0036_f4a414c4155d.png     other      0036_f4a414c4155d.png
1  lunar_new_year  0008_8843ddf7b700.jpg     other  tet_0008_8843ddf7b700.jpg
2  lunar_new_year  0029_5402f43a6653.jpg     other  tet_0029_5402f43a6653.jpg
3  lunar_new_year  0034_a077e240af5b.jpg     other  tet_0034_a077e240af5b.jpg
4  lunar_new_year  0036_adb92c35a73c.jpg     other  tet_0036_adb92c35a73c.jpg

## SigLIP2: Semantic Index

### Semantic Index

In [ ]:
import fiftyone.zoo as foz
from fiftyone import ViewField as F

from z_photos.fiftyone import register_models

register_models()
CLASSES = sorted(bronze_dataset.distinct("ground_truth.label"))
siglip2_zs = foz.load_zoo_model("z-photos-siglip2-so400m-patch16-384", classes=CLASSES)

unprocessed_view = bronze_dataset.match(~F("siglip2_embeddings").exists())
if len(unprocessed_view) > 0:
    print(f"SigLIP2 embeddings: {len(unprocessed_view)} còn lại")
    unprocessed_view.compute_embeddings(
        siglip2_zs, embeddings_field="siglip2_embeddings"
    )
else:
    print("SigLIP2 embeddings: OK")

SigLIP2 embeddings: OK


In [ ]:
import fiftyone.brain as fob

if "siglip2_viz" in bronze_dataset.list_brain_runs():
    bronze_dataset.delete_brain_run("siglip2_viz")
fob.compute_visualization(
    bronze_dataset, embeddings="siglip2_embeddings", brain_key="siglip2_viz"
)
print("SigLIP2 visualization OK.")

if "siglip2_sim" in bronze_dataset.list_brain_runs():
    bronze_dataset.delete_brain_run("siglip2_sim")
siglip2_sim = fob.compute_similarity(
    bronze_dataset,
    model="z-photos-siglip2-so400m-patch16-384",
    embeddings="siglip2_embeddings",
    brain_key="siglip2_sim",
)
print(f"SigLIP2 similarity index ok. supports_prompts={siglip2_sim.supports_prompts}")

Generating visualization...
UMAP( verbose=True)
Fri Mar 13 22:57:48 2026 Construct fuzzy simplicial set
Fri Mar 13 22:57:48 2026 Finding Nearest Neighbors
Fri Mar 13 22:57:48 2026 Finished Nearest Neighbor Search
Fri Mar 13 22:57:48 2026 Construct embedding


Epochs completed:   0%|            0/500 [00:00]

	completed  0  /  500 epochs
	completed  50  /  500 epochs
	completed  100  /  500 epochs
	completed  150  /  500 epochs
	completed  200  /  500 epochs
	completed  250  /  500 epochs
	completed  300  /  500 epochs
	completed  350  /  500 epochs
	completed  400  /  500 epochs
	completed  450  /  500 epochs
Fri Mar 13 22:57:48 2026 Finished embedding
SigLIP2 visualization OK.
SigLIP2 similarity index ok. supports_prompts=True


### Zero-shot Classification

In [ ]:
from collections import Counter

import pandas as pd
from fiftyone import ViewField as F

unprocessed_view = bronze_dataset.match(~F("siglip2_pred").exists())
if len(unprocessed_view) > 0:
    print(f"SigLIP2 zero-shot: {len(unprocessed_view)} còn lại")
    unprocessed_view.apply_model(siglip2_zs, label_field="siglip2_pred")
else:
    print("SigLIP2 zero-shot: OK")

gt_labels = bronze_dataset.values("ground_truth.label")
pred_labels = bronze_dataset.values("siglip2_pred.label")
agree = sum(g == p for g, p in zip(gt_labels, pred_labels, strict=True))
total = len(bronze_dataset)
print(f"Agreement: {agree}/{total} = {agree / total * 100:.1f}%")
# Disagreements theo class
disagree_by_class: dict[str, list[str]] = {}
for g, p in zip(gt_labels, pred_labels, strict=False):
    if g != p:
        disagree_by_class.setdefault(g, []).append(p)

rows = []
for gt_cls in sorted(disagree_by_class):
    preds = Counter(disagree_by_class[gt_cls])
    for pred_cls, cnt in preds.most_common():
        rows.append({"ground_truth": gt_cls, "siglip2_pred": pred_cls, "count": cnt})

df_disagree = pd.DataFrame(rows)
print("SigLIP2 disagreements (ground_truth ≠ prediction):")
display(df_disagree.style.hide(axis="index"))

SigLIP2 zero-shot: OK
Agreement: 39/321 = 12.1%
SigLIP2 disagreements (ground_truth ≠ prediction):


ground_truth,siglip2_pred,count
baby_playing,lunar_new_year,48
gathering,lunar_new_year,29
nature,lunar_new_year,74
other,lunar_new_year,80
other,baby_playing,2
trekking,lunar_new_year,49


## C-RADIOv4: Visual QA Index

C-RADIOv4 là **multi-teacher distilled model** (SigLIP2 + DINOv3 + SAM3) → embedding rất mạnh về visual detail.

Dùng cho:
- `compute_near_duplicates` — ảnh **rất giống nhau về thị giác** (crop, resize, re-composition)
- `compute_leaky_splits` — phát hiện data leakage train → test
- `compute_uniqueness` — phân tích cụm thị giác

In [ ]:
import fiftyone.zoo as foz
from fiftyone import ViewField as F

foz.register_zoo_model_source("https://github.com/harpreetsahota204/CRADIOv4")
radio_model = foz.load_zoo_model("nv_labs/c-radio_v4-so400m")

unprocessed_view = bronze_dataset.match(~F("radio_embeddings").exists())
if len(unprocessed_view) > 0:
    print(f"RADIO embeddings: {len(unprocessed_view)} còn lại")
    unprocessed_view.compute_embeddings(
        radio_model, embeddings_field="radio_embeddings", num_workers=0
    )
else:
    print("RADIO embeddings: OK")

Loading weights:   0%|          | 0/330 [00:00<?, ?it/s]

RADIO embeddings: OK


### Near Duplicates

In [ ]:
# Build similarity index từ RADIO embeddings (dùng để near-dups + leaky splits)
if "radio_sim" in bronze_dataset.list_brain_runs():
    bronze_dataset.delete_brain_run("radio_sim")
radio_sim = fob.compute_similarity(
    bronze_dataset, embeddings="radio_embeddings", brain_key="radio_sim"
)

# Near duplicates: ảnh rất giống nhau về mặt thị giác
near_dup_index = fob.compute_near_duplicates(
    bronze_dataset,
    similarity_index=radio_sim,
    threshold=0.1,  # cosine distance < 0.1 → very similar visually
)

dup_ids = near_dup_index.duplicate_ids
print(f"Near duplicate samples (C-RADIOv4, threshold=0.1): {len(dup_ids)}")
rate = len(dup_ids) / len(bronze_dataset) * 100
print(f"Near-dup rate: {len(dup_ids)}/{len(bronze_dataset)} = {rate:.1f}%")

if dup_ids:
    dup_samples = bronze_dataset.select(dup_ids)
    dup_label_counts = Counter(dup_samples.values("ground_truth.label"))
    dup_tag_counts = Counter([t for ts in dup_samples.values("tags") for t in ts])
    df_nd = pd.DataFrame(
        {
            "class": list(dup_label_counts.keys()),
            "near_dup_count": list(dup_label_counts.values()),
        }
    ).set_index("class")
    display(df_nd.style.format("{:d}"))

Computing duplicate samples...


Duplicates computation complete


Near duplicate samples (C-RADIOv4, threshold=0.1): 17
Near-dup rate: 17/321 = 5.3%


### Leaky Splits Detection

Tìm ảnh train và test quá giống nhau về nội dung thị giác → **data leakage** → metric bị inflate.  
Dataset nhỏ (~261 train / 60 test) → vài ảnh trùng có thể làm accuracy tăng giả tạo.

In [ ]:
# Detect leaky splits: samples in test too similar to train
leaky_index = fob.compute_leaky_splits(
    bronze_dataset, splits=["train", "test"], similarity_index=radio_sim
)

if leaks := leaky_index.leaks_view():
    n_test = len(leaks.match_tags("test"))
    # Analytical signal: how much of our test set is compromised?
    print(f"Leakage: {n_test} test samples ({n_test / len(test_view):.1%})")
    for s in leaks:
        split = "train" if "train" in s.tags else "test"
        print(f"  [{split}] {s.ground_truth.label}: {Path(s.filepath).name}")
    leaky_index.tag_leaks("leaky")
else:
    print("No leakage detected.")

Leakage: 10 test samples (16.7%)
  [test] baby_playing: 0025_2a620b8cb4c2.jpg
  [train] baby_playing: 0033_dc8d732281ba.jpg
  [train] lunar_new_year: 0041_139944057583.jpg
  [test] lunar_new_year: 0001_b9ec44a6a1b8.jpg
  [train] nature: 0001_2debb1a58275.jpg
  [test] nature: 0003_95092aadfa4f.jpg
  [train] nature: 0021_8527763b6ab0.jpg
  [test] nature: 0012_47e5ed29fc3c.jpg
  [train] trekking: 0039_8015ea087fa1.jpg
  [test] trekking: 0046_9ecb742b9cf9.webp
  [train] baby_playing: 0003_4b315d3e7d10.jpg
  [test] baby_playing: 0009_794754ce39b6.jpg
  [train] other: 0043_7c849c1b18ba.jpg
  [test] baby_playing: 0043_7c849c1b18ba.jpg
  [train] gathering: 0346_d79805bc0361.png
  [test] lunar_new_year: 0346_d79805bc0361.png
  [train] nature: 0018_025ab34fb807.jpg
  [test] other: 0276_a544a6e41678.jpg
  [train] nature: 0180_318a682da5de.jpg
  [test] nature: 0005_f2807771fe4d.jpg


### Uniqueness & Representativeness

In [ ]:
import numpy as np
import pandas as pd

# 1. Compute uniqueness
fob.compute_uniqueness(
    bronze_dataset,
    similarity_index=radio_sim,
    uniqueness_field="radio_uniqueness",
)

u_arr = np.array(
    [v for v in bronze_dataset.values("radio_uniqueness") if v is not None]
)
print(
    "Uniqueness (C-RADIOv4):",
    f"min={u_arr.min():.3f} | max={u_arr.max():.3f} | mean={u_arr.mean():.3f}",
)


# 2. Extract & Display Insights using DataFrame
def get_uniqueness_samples(reverse_sort, category_desc):
    """Lấy top 5 mẫu theo chiều sắp xếp và định dạng thành dict với tên cột rõ ràng."""
    samples = bronze_dataset.sort_by("radio_uniqueness", reverse=reverse_sort).limit(5)
    return [
        {
            "Data_Category": category_desc,
            "Uniqueness_Score": f"{s.radio_uniqueness:.3f}",
            "Ground_Truth": s.ground_truth.label,
            "Dataset_Tags": ", ".join(s.tags) if s.tags else "",
            "File_Name": s.filepath.split("/")[-1],
        }
        for s in samples.iter_samples()
    ]


# Gom nhóm: Điểm cao nhất (Most Unique) và Điểm thấp nhất (Least Unique)
analysis_rows = get_uniqueness_samples(
    True, "Most Unique (Potential OOD)"
) + get_uniqueness_samples(False, "Least Unique (Redundant)")

df_analysis = pd.DataFrame(analysis_rows)
display(df_analysis.style.hide(axis="index"))

# 3. Compute representativeness
fob.compute_representativeness(
    bronze_dataset,
    embeddings="radio_embeddings",
    representativeness_field="radio_representativeness",
)

Retrieving embeddings from similarity index...
Computing uniqueness...
Uniqueness computation complete
Uniqueness (C-RADIOv4): min=0.167 | max=1.000 | mean=0.528


Data_Category,Uniqueness_Score,Ground_Truth,Dataset_Tags,File_Name
Most Unique (Potential OOD),1.000,other,train,0205_24b529147cbb.jpg
Most Unique (Potential OOD),0.979,other,test,0290_7eeca90a0455.webp
Most Unique (Potential OOD),0.977,other,train,0388_98b82a8b3c46.jpg
Most Unique (Potential OOD),0.971,other,train,0289_fa327ce24889.jpg
Most Unique (Potential OOD),0.960,other,train,0364_9a6f26073a1b.jpg
Least Unique (Redundant),0.167,nature,"train, leaky",0021_8527763b6ab0.jpg
Least Unique (Redundant),0.172,nature,train,0025_94ef770219a5.jpg
Least Unique (Redundant),0.173,nature,train,0011_3dba24066d0e.jpg
Least Unique (Redundant),0.180,nature,train,0019_a24c5fbca5b8.jpg
Least Unique (Redundant),0.189,other,train,tet_0036_adb92c35a73c.jpg


Computing representativeness...
Computing clusters for 321 embeddings; this may take awhile...
Representativeness computation complete


# Future Works